# Artifact 24: integer-scaled mapped-area periods and mesh verification

This executable notebook assembles four mapped-area period calculations:

- the **red triangular-disk family**;
- the **green Abel–Wick lobe**;
- the **yellow Abel–Wick lobe**;
- the **blue Abel–Wick lobe**.

It contains:

1. exact integer-scaled period coefficients;
2. reconstruction of mapped area by integrating the period series;
3. direct pullback quadrature;
4. mapped-triangle mesh convergence;
5. the interactive two-panel Hamilton–Abel $Q/R$ coordinate-net depiction, including the pink missing cubic;
6. four mapped mesh depictions for red, green, yellow, and blue.

The exact prefixes currently included are **90 red terms** and **100 terms for each Abel–Wick projection**. All arithmetic used to produce these sequence tables was rational/integer arithmetic, not modular reconstruction.

## Bounded action–angle and Gram-matrix formulation

For each bounded oscillation disk, use canonical coordinates

\[
(X,Y)=\sqrt{2\lambda}(\cos\phi,\sin\phi),
\qquad dX\wedge dY=d\lambda\wedge d\phi.
\]

For source-plane embedding $E_i$ and polynomial map $F$, define

\[
\mathcal R_i(\lambda,\phi)=F(E_i(\lambda,\phi)),
\qquad
G_i=(DE_i)^T(DF)^TDF(DE_i).
\]

The mapped-area density is

\[
g_i(\lambda,\phi)=\sqrt{\det G_i(\lambda,\phi)}.
\]

If $\lambda_\alpha(\phi)$ is the bounded root of the energy equation, then

\[
\mathcal A_i(\alpha)
=\int_0^{2\pi}\int_0^{\lambda_\alpha(\phi)}g_i(\lambda,\phi)\,d\lambda\,d\phi,
\]

and the mapped-area period is

\[
\mathcal A_i'(\alpha)
=\int_0^{2\pi}g_i(\lambda_\alpha,\phi)
\,\partial_\alpha\lambda_\alpha\,d\phi.
\]

We normalize

\[
\Phi_i(s)=\Psi_i'(s)=\sum_{k\ge0}b_{i,k}s^k,
\qquad
\Psi_i(s)=\frac{\mathcal A_i(s)}{\pi J_i(0)}.
\]

The recorded integer sequence is

\[
A_{i,k}=b_{i,k}C_i^k\in\mathbb Z,
\]

so normalized area is reconstructed directly by

\[
\Psi_i(s)=\sum_{k\ge0}\frac{A_{i,k}}{k+1}\frac{s^{k+1}}{C_i^k}.
\]

In [ ]:
from __future__ import annotations

from pathlib import Path
from io import StringIO
import base64, zlib, math, importlib.util, sys

import numpy as np
import pandas as pd
import sympy as sp
import mpmath as mp
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from numpy.polynomial.legendre import leggauss

mp.mp.dps = 80

# The complete exact tables are embedded so the notebook does not depend on
# a separate CSV merely to print or evaluate the sequences.
_SEQUENCE_BLOB = "..."
_RED_VALIDATION_BLOB = "eNp9lM1uGzEMhO99lsWCpPgjPk0QoAaaQw91mvfvJzmO0xbtaddjcjQcjvb78ePt+ev1+efb9fL0fL08H6+X68vl9cnvL/P+onZ/Mzm+X16/PVn6z+vL7V1VHj881B7/UCb8+iKn2KGnuWR4iZiVaTyg7MnD5n+LPqD6B2Qx1b2GVc6ac0Eje2gH3GJSY0NdNkxzIlt03rgoGuITubrExjFOabFoq4guSP0DkqQ55ROyi9L177Z/ICFR0QOVlqM30RRrtKlqUXuDqtLTKwuFVRvqkdKT7uk2J1r1yFMHR6A/zD283xFF4zpjTP9U4xlzoPT3Lsr+5IkBgtJ2EW9TmhaiDB+FBixoLN9tw/F0Oc8C5kZK0DzHckUdmazRkB9SLI7p1bVukM+BF9gK5B9VELjlbPNPjRSJ4OPfXH2onowqABJzKtbuKlQTFMZun6UbKsXV1u7hbOTGxdgVWbRmBHL9sHEq20kxhULYwoYQlhU1ncki36vAxopXpo17I2mg0FXyQaWsr0iiHWYnQpUk8fAp+zwiNSyV+GbVvPV5kA9T0VKzG5TkhLBApuHL2zyGnb0z3LHG17EQWHF0hhZz7Rp8IVFchyzkb6RLuESigziF34kYsNVJbSwEU7GWC9PsfgFr85lw0cNV2ETkW4XprNv6xr1COoVITzNkEuA6l8+DVAk5It7HyHNli6CY9sTbVaQSweBKgHTGRvg+8C3IIlIxNmQ+1BESBMgXT4XhLalvrntsqDHcCKsjYk8ME3ePLUZiwzofIvGxONDZ8eUXJXYT8A=="
_AW_VALIDATION_BLOB = "eNqtlEtuGzEQRPc6y0To/+c0gpxMEgNyAkgJAt8+xeHAlu2tNwJVQzaru1/z+/np8fK83JZvj9f165/T+bqel9t6fVxv2/rERG/+nx9up/V6/X1dntbbz5MVzQV32VwVxb7iEqO75fvDU7yulykeflzX9ddCRxL8cCeTdYelmXJ9lHqpo0pESpRFVaStXzi3jZ6eQUppkTSlyEhnEXdt2cORkJcHIar2LiFMpjiHtS96xJkWkSxRCsEN5MMiETeps7l5cr96H19dlInLELs5lD9KsfjRnZnhSFk8S+E9xkZcl8NSSEmSbWeFkS8nMh9ZTKlCSIq95jGRbheDqWSp2j1SeEeJkJpMCcvEbSpF1Ifn9XL5/W/7MlwKU3VIZqCu7PZeotEIPsKPmCvsFeyIwPq2ldqLNKrhrXwe9ojKbDJp7j0eaxMjy0KAXYLN0DSy7ETVcbuio6woLkvcVT01sD0ahfN7+yM9Yx8+mwQ4ojkfJB6xFXGLAAKqk7JB49tOBRsBpxqqmzEcHg22QuYenlPCadcYe4DIfgUFjLV3pdiSR2IOcZAqkWz39qNCwRqW9cb++I79ieYNYkZd472EiqP4QF+qW2WUEAjuxe9G2ztr4KOQN8kN97QOyIzmrjAZFmqgo1NBQh2ELBW92336GJRSsVmJKSXwGmmpHB4uf9fPxQbzW1S0mdq5KRZCAPU7bgB9osz33Bihqmaxc+MOhgjDg8fiLTjUymrg7NX+J2FT1CnMus3ijg0eEoV35Vds4JTDS+oOGwciHuYTG5wbY++Oxr3DBvVX9lfrn4MMbrIEI7zzgdcRs+VRNB/dSQyG1QLTofKCDF6jasNYyAsfxm1i6JC9SI2aGJyVHP4DWMRnjA=="

def _unpack_csv(blob: str) -> pd.DataFrame:
    text = zlib.decompress(base64.b64decode(blob)).decode("utf-8")
    return pd.read_csv(StringIO(text), dtype=str)

sequence_table = _unpack_csv(_SEQUENCE_BLOB)
red_precomputed = _unpack_csv(_RED_VALIDATION_BLOB)
aw_precomputed = _unpack_csv(_AW_VALIDATION_BLOB)

SCALES = {
    "red": 235_651_734,
    "green": 71_912_664,
    "yellow": 943_148_703_570_448_617_216,
    "blue": 943_148_703_570_448_617_216,
}
CENTER_DENSITIES = {
    "red": mp.sqrt(6267) / 4,
    "green": mp.sqrt(577) / 8,
    "yellow": mp.sqrt(mp.mpf(80014294261) / mp.mpf(17252352)),
    "blue": mp.sqrt(mp.mpf(80014294261) / mp.mpf(17252352)),
}

assert sequence_table["is_integer"].str.lower().eq("true").all()
assert set(sequence_table.family) == {"red", "green", "yellow", "blue"}
print(sequence_table.groupby("family").size().rename("exact terms"))

In [ ]:
def family_sequence(family: str) -> pd.DataFrame:
    out = sequence_table[sequence_table.family == family].copy()
    out["k"] = out["k"].astype(int)
    return out.sort_values("k")


def print_sequence(family: str, count: int = 8) -> None:
    data = family_sequence(family).head(count)
    values = data["scaled_integer"].tolist()
    print(f"{family}: C = {SCALES[family]}")
    print(",\n".join(values) + (", ..." if len(family_sequence(family)) > count else ""))
    print()


for name in ("red", "green", "yellow", "blue"):
    print_sequence(name, 8 if name != "yellow" else 6)

# Yellow and blue were generated independently.
y = family_sequence("yellow")["scaled_integer"].tolist()
b = family_sequence("blue")["scaled_integer"].tolist()
assert y == b
print("Exact yellow/blue equality through", len(y), "terms: PASS")

## Exact homogeneous density recurrence

After a bounded coordinate rationalization, write the normalized Gram determinant and its square root as homogeneous sums

\[
\widetilde Q_i=\sum_{k=0}^{22}q_{i,k},
\qquad
\sqrt{\widetilde Q_i}=\sum_{n\ge0}j_{i,n}.
\]

The homogeneous pieces satisfy

\[
\boxed{
 j_{i,n}=-\frac1{2n}
 \sum_{k=1}^{\min(22,n)}(2n-3k)q_{i,k}j_{i,n-k}.
}
\]

This recurrence generated the exact density expansion. Angular moments over the bounded disk then generated the rational period coefficients. The observed integer scales are

\[
C_R=235651734,
\qquad C_G=71912664,
\qquad C_Y=C_B=943148703570448617216.
\]

The notebook treats these as strongly tested arithmetic observations, not as all-orders integrality theorems.

In [ ]:
def normalized_area_from_sequence(family: str, s, terms: int | None = None):
    data = family_sequence(family)
    if terms is not None:
        data = data.head(terms)
    s = mp.mpf(str(s))
    C = mp.mpf(SCALES[family])
    total = mp.mpf("0")
    for row in data.itertuples(index=False):
        k = int(row.k)
        A = mp.mpf(row.scaled_integer)
        total += A * s**(k + 1) / ((k + 1) * C**k)
    return total


def mapped_area_from_sequence(family: str, s, terms: int | None = None):
    return mp.pi * CENTER_DENSITIES[family] * normalized_area_from_sequence(
        family, s, terms
    )

# Representative sequence-reconstructed areas.
for family, level in [("red", 0.4), ("green", 0.05),
                      ("yellow", 0.005), ("blue", 0.005)]:
    print(family, level, mp.nstr(mapped_area_from_sequence(family, level), 25))

## Precomputed independent checks

These tables provide immediate reference values. Later cells recompute quadrature and meshes from the polynomial map when the Artifact 24 repository is available.

In [ ]:
display(red_precomputed)
display(aw_precomputed[[
    "family", "s", "direct_area", "series_area_100",
    "mesh_480", "mesh_1984", "mesh_8064", "mesh_18240"
]])

# Interactive Hamilton–Abel depiction

The next cells are the supplied two-panel coordinate-net construction, separated into notebook stages. They retain its algebraic verification, constant-$Q$ and constant-$R$ curves, finite preimages, and the pink missing cubic.

## Bootstrap and package loading

In [ ]:
from pathlib import Path
import sys
import numpy as np
import plotly.graph_objects as go

# ============================================================
# Bootstrap: locate the artifact24 package
# ============================================================

HERE = Path.cwd().resolve()

candidates = []
for base in (HERE, *HERE.parents):
    candidates.extend([
        base,
        base / "24-jupyter-testing",
        base / "Artifacts" / "24-jupyter-testing",
    ])

PROJECT_ROOT = next(
    (
        candidate
        for candidate in candidates
        if (candidate / "artifact24" / "geometry.py").is_file()
        and (candidate / "artifact24" / "plots.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate the artifact24 package.\n"
        f"Current directory: {HERE}\n\n"
        "The notebook must be inside the Artifacts repository, or beside "
        "the 24-jupyter-testing folder."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from artifact24.geometry import polynomial_map
from artifact24.plots import build_interactive_figure, source_display

print("Artifact 24 loaded from:", PROJECT_ROOT)



## Coordinate-net choices and helper functions

In [ ]:
# ============================================================
# User choices
# ============================================================

Q_LEVELS = [-6.0, -4.0, -2.0, 0.0, 2.0, 4.0, 6.0]
R_LEVELS = [-1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]

BLACK = "#111111"
PINK = "#ff00aa"

# ============================================================
# Geometry / algebra helpers
# ============================================================

def discriminant(P, Q, R):
    return Q*Q - 16.0*P - Q**3 * R + 18.0*P*Q*R - 27.0*(P**2)*(R**2)

def discriminant_error(P, Q, R):
    """
    Return both absolute and scale-aware residuals.

    The discriminant is a cancellation of five polynomial terms.  Its
    absolute roundoff can be around 1e-9 even when the relative error is near
    machine precision, so verification must use the scaled residual.
    """
    terms = (
        Q*Q,
        -16.0*P,
        -Q**3 * R,
        18.0*P*Q*R,
        -27.0*(P**2)*(R**2),
    )
    residual = abs(sum(terms))
    scale = max(1.0, sum(abs(term) for term in terms))
    return residual, residual / scale

def finite_preimage_from_rho_c(rho, c):
    """
    Finite preimage surface parameterization in RAW source coordinates.
    """
    den = 3.0 * c * rho - 2.0
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        x = 2.0 * c / den**2
        y = rho * (8.0 - 9.0 * c * rho) / 2.0
        z = (
            -9.0
            * rho**2
            * den**2
            * (9.0 * c**2 * rho**2 - 24.0 * c * rho + 14.0)
            / 8.0
        )
    return x, y, z, den

def _trace_bounds(figure, scene_name, coordinate):
    chunks = []
    for trace in figure.data:
        if getattr(trace, "scene", "scene") != scene_name:
            continue
        values = getattr(trace, coordinate, None)
        if values is None:
            continue
        try:
            values = np.asarray(values, dtype=float).reshape(-1)
        except (TypeError, ValueError):
            continue
        values = values[np.isfinite(values)]
        if values.size:
            chunks.append(values)

    if not chunks:
        raise RuntimeError(f"No finite {coordinate}-values found in {scene_name}")

    values = np.concatenate(chunks)
    lo = float(values.min())
    hi = float(values.max())
    pad = 0.03 * max(hi - lo, 1.0)
    return lo - pad, hi + pad

def _clip3(x, y, z, xr, yr, zr):
    mask = (
        np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
        & (x >= xr[0]) & (x <= xr[1])
        & (y >= yr[0]) & (y <= yr[1])
        & (z >= zr[0]) & (z <= zr[1])
    )
    return (
        np.where(mask, x, np.nan),
        np.where(mask, y, np.nan),
        np.where(mask, z, np.nan),
    )



## Built-in algebraic verification

In [ ]:
# ============================================================
# Verification
# ============================================================

rho_samples_Q = np.array([-3.0, -1.7, -0.8, -0.3, 0.35, 0.9, 1.8, 3.1], dtype=float)
rho_samples_R = np.array([-3.2, -2.0, -1.2, -0.6, 0.5, 1.1, 2.1, 3.0], dtype=float)

max_inverse_residual = 0.0
max_surface_absolute_residual = 0.0
max_surface_scaled_residual = 0.0
max_fixed_coordinate_residual = 0.0
max_pink_absolute_error = 0.0
max_pink_scaled_error = 0.0
qr_lattice_real = 0
qr_lattice_total = 0

# Constant-Q family verification
for q0 in Q_LEVELS:
    for rho in rho_samples_Q:
        if abs(rho) < 1e-12:
            continue

        P = rho * (q0 - rho) / 3.0
        Q = q0
        R = (4.0 * rho - q0) / (3.0 * rho**2)

        surface_abs, surface_scaled = discriminant_error(P, Q, R)
        max_surface_absolute_residual = max(
            max_surface_absolute_residual, surface_abs
        )
        max_surface_scaled_residual = max(
            max_surface_scaled_residual, surface_scaled
        )
        max_fixed_coordinate_residual = max(
            max_fixed_coordinate_residual, abs(Q - q0)
        )

        x, y, z, den = finite_preimage_from_rho_c(np.array([rho]), np.array([R]))
        if abs(den[0]) < 1e-7:
            continue
        mapped = polynomial_map(np.array([[x[0], y[0], z[0]]], dtype=float))[0]
        target = np.array([P, Q, R], dtype=float)

        scale = max(1.0, np.max(np.abs(target)))
        max_inverse_residual = max(
            max_inverse_residual,
            np.max(np.abs(mapped - target)) / scale
        )

# Constant-R family verification
for r0 in R_LEVELS:
    for rho in rho_samples_R:
        P = rho**2 - r0 * rho**3
        Q = 4.0 * rho - 3.0 * r0 * rho**2
        R = r0

        surface_abs, surface_scaled = discriminant_error(P, Q, R)
        max_surface_absolute_residual = max(
            max_surface_absolute_residual, surface_abs
        )
        max_surface_scaled_residual = max(
            max_surface_scaled_residual, surface_scaled
        )
        max_fixed_coordinate_residual = max(
            max_fixed_coordinate_residual, abs(R - r0)
        )

        x, y, z, den = finite_preimage_from_rho_c(np.array([rho]), np.array([r0]))
        if abs(den[0]) < 1e-7:
            continue
        mapped = polynomial_map(np.array([[x[0], y[0], z[0]]], dtype=float))[0]
        target = np.array([P, Q, R], dtype=float)

        scale = max(1.0, np.max(np.abs(target)))
        max_inverse_residual = max(
            max_inverse_residual,
            np.max(np.abs(mapped - target)) / scale
        )

# Pink missing curve verification
pink_rho = np.array([-4.0, -2.0, -0.7, 0.5, 1.6, 3.0], dtype=float)
for rho in pink_rho:
    P = rho**2 / 3.0
    Q = 2.0 * rho
    R = 2.0 / (3.0 * rho)
    pink_abs, pink_scaled = discriminant_error(P, Q, R)
    max_pink_absolute_error = max(max_pink_absolute_error, pink_abs)
    max_pink_scaled_error = max(max_pink_scaled_error, pink_scaled)

# Q/R lattice: each pair (Q,R) should generically meet surface in 0,1,2 real points.
# For our chosen modest lattice this mainly checks that the chosen levels do produce
# intersections around the central visible figure.
for q0 in Q_LEVELS:
    for r0 in R_LEVELS:
        qr_lattice_total += 1
        # Solve 3 r rho^2 - 4 rho + q = 0
        if abs(r0) < 1e-14:
            # Then q = 4 rho, exactly one real rho
            qr_lattice_real += 1
        else:
            disc = 16.0 - 12.0 * r0 * q0
            if disc >= 0.0:
                qr_lattice_real += 1

assert max_inverse_residual < 1e-8, max_inverse_residual
assert max_surface_scaled_residual < 1e-12, max_surface_scaled_residual
assert max_fixed_coordinate_residual < 1e-12, max_fixed_coordinate_residual
assert max_pink_scaled_error < 1e-12, max_pink_scaled_error

print("Verification passed")
print(f"  max inverse scaled residual:     {max_inverse_residual:.3e}")
print(f"  max surface absolute residual:   {max_surface_absolute_residual:.3e}")
print(f"  max surface scaled residual:     {max_surface_scaled_residual:.3e}")
print(f"  max fixed-coordinate residual:   {max_fixed_coordinate_residual:.3e}")
print(f"  max pink absolute error:         {max_pink_absolute_error:.3e}")
print(f"  max pink scaled error:           {max_pink_scaled_error:.3e}")
print(f"  Q/R lattice pairs with real intersections: {qr_lattice_real}/{qr_lattice_total}")



## Build the two-panel figure and overlays

In [ ]:
# ============================================================
# Build the original Artifact 24 figure and remove markers
# ============================================================

fig_base = build_interactive_figure(
    red_count=12,
    auxiliary_count=8,
    samples=420,
    show_background=True,
)

filtered = []
for trace in fig_base.data:
    mode = str(getattr(trace, "mode", ""))
    if trace.type == "scatter3d" and mode == "markers":
        continue
    filtered.append(trace)

fig1 = go.Figure(data=filtered, layout=fig_base.layout)

# Slightly strengthen surviving colored line traces
for trace in fig1.data:
    if trace.type == "scatter3d" and "lines" in str(getattr(trace, "mode", "")):
        old_width = getattr(getattr(trace, "line", None), "width", None)
        old_width = 2.0 if old_width is None else float(old_width)
        trace.line.width = max(old_width, 3.0)

# Preserve the original plotting windows
domain_bounds = {
    "x": _trace_bounds(fig1, "scene", "x"),
    "y": _trace_bounds(fig1, "scene", "y"),
    "z": _trace_bounds(fig1, "scene", "z"),
}
range_bounds = {
    "x": _trace_bounds(fig1, "scene2", "x"),  # P
    "y": _trace_bounds(fig1, "scene2", "y"),  # Q
    "z": _trace_bounds(fig1, "scene2", "z"),  # R
}

x_range = domain_bounds["x"]
y_range = domain_bounds["y"]
z_range = domain_bounds["z"]

P_range = range_bounds["x"]
Q_range = range_bounds["y"]
R_range = range_bounds["z"]

rho_line = np.linspace(-4.0, 4.0, 3200)
rho_abs = np.geomspace(0.04, 8.0, 4200)

# ============================================================
# Constant R curves (range + finite preimages)
# ============================================================

first_R_range = True
first_R_domain = True

for R0 in R_LEVELS:
    rho = rho_line
    c = np.full_like(rho, R0, dtype=float)

    # Range curve
    P = rho**2 - c * rho**3
    Q = 4.0 * rho - 3.0 * c * rho**2
    R = c

    P, Q, R = _clip3(P, Q, R, P_range, Q_range, R_range)

    if np.isfinite(P).sum() >= 2:
        fig1.add_trace(
            go.Scatter3d(
                x=P, y=Q, z=R,
                mode="lines",
                line=dict(color=BLACK, width=2),
                name="constant R" if first_R_range else "constant R ",
                legendgroup="const-R-range",
                showlegend=first_R_range,
                hovertemplate=(
                    f"<b>constant R={R0:g}</b>"
                    "<br>P=%{x:.5g}"
                    "<br>Q=%{y:.5g}"
                    "<extra></extra>"
                ),
                scene="scene2",
            )
        )
        first_R_range = False

    # Domain finite preimage
    X, Y, Z, den = finite_preimage_from_rho_c(rho, c)
    away = np.abs(den) >= 0.04
    X = np.where(away, X, np.nan)
    Y = np.where(away, Y, np.nan)
    Z = np.where(away, Z, np.nan)

    raw_pts = np.column_stack([X, Y, Z])
    disp_pts = source_display(raw_pts)

    XD = disp_pts[:, 0]
    YD = disp_pts[:, 1]
    ZD = disp_pts[:, 2]

    XD, YD, ZD = _clip3(XD, YD, ZD, x_range, y_range, z_range)

    if np.isfinite(XD).sum() >= 2:
        fig1.add_trace(
            go.Scatter3d(
                x=XD, y=YD, z=ZD,
                mode="lines",
                line=dict(color=BLACK, width=2),
                name="preimage of constant R" if first_R_domain else "preimage of constant R ",
                legendgroup="const-R-domain",
                showlegend=first_R_domain,
                hovertemplate=(
                    f"<b>preimage of R={R0:g}</b>"
                    "<br>display x=%{x:.5g}"
                    "<br>display y=%{y:.5g}"
                    "<br>display z=%{z:.5g}"
                    "<extra></extra>"
                ),
                scene="scene",
            )
        )
        first_R_domain = False

# ============================================================
# Constant Q curves (range + finite preimages)
# ============================================================

first_Q_range = True
first_Q_domain = True

for Q0 in Q_LEVELS:
    for sign in (+1.0, -1.0):
        rho = sign * rho_abs

        # Range curve
        P = rho * (Q0 - rho) / 3.0
        Q = np.full_like(rho, Q0, dtype=float)
        R = (4.0 * rho - Q0) / (3.0 * rho**2)

        P, Q, R = _clip3(P, Q, R, P_range, Q_range, R_range)

        if np.isfinite(P).sum() >= 2:
            fig1.add_trace(
                go.Scatter3d(
                    x=P, y=Q, z=R,
                    mode="lines",
                    line=dict(color=BLACK, width=3),
                    name="constant Q" if first_Q_range else "constant Q ",
                    legendgroup="const-Q-range",
                    showlegend=first_Q_range,
                    hovertemplate=(
                        f"<b>constant Q={Q0:g}</b>"
                        "<br>P=%{x:.5g}"
                        "<br>R=%{z:.5g}"
                        "<extra></extra>"
                    ),
                    scene="scene2",
                )
            )
            first_Q_range = False

        # Domain finite preimage
        c = (4.0 * rho - Q0) / (3.0 * rho**2)
        X, Y, Z, den = finite_preimage_from_rho_c(rho, c)

        away = np.abs(den) >= 0.04
        X = np.where(away, X, np.nan)
        Y = np.where(away, Y, np.nan)
        Z = np.where(away, Z, np.nan)

        raw_pts = np.column_stack([X, Y, Z])
        disp_pts = source_display(raw_pts)

        XD = disp_pts[:, 0]
        YD = disp_pts[:, 1]
        ZD = disp_pts[:, 2]

        XD, YD, ZD = _clip3(XD, YD, ZD, x_range, y_range, z_range)

        if np.isfinite(XD).sum() >= 2:
            fig1.add_trace(
                go.Scatter3d(
                    x=XD, y=YD, z=ZD,
                    mode="lines",
                    line=dict(color=BLACK, width=3),
                    name="preimage of constant Q" if first_Q_domain else "preimage of constant Q ",
                    legendgroup="const-Q-domain",
                    showlegend=first_Q_domain,
                    hovertemplate=(
                        f"<b>preimage of Q={Q0:g}</b>"
                        "<br>display x=%{x:.5g}"
                        "<br>display y=%{y:.5g}"
                        "<br>display z=%{z:.5g}"
                        "<extra></extra>"
                    ),
                    scene="scene",
                )
            )
            first_Q_domain = False

# ============================================================
# Pink missing cubic in the range only
#
# Draw it twice:
#   1. a broad white under-stroke separating it from black lines;
#   2. a narrower saturated pink stroke.
#
# Both traces are appended AFTER every black and colored curve.
# ============================================================

rho_missing_abs = np.geomspace(1.0e-5, 1.0e5, 50000)
pink_branch_data = []

for sign in (+1.0, -1.0):
    rho = sign * rho_missing_abs

    P = rho**2 / 3.0
    Q = 2.0 * rho
    R = 2.0 / (3.0 * rho)

    P, Q, R = _clip3(P, Q, R, P_range, Q_range, R_range)

    visible = np.isfinite(P) & np.isfinite(Q) & np.isfinite(R)
    if np.count_nonzero(visible) >= 2:
        pink_branch_data.append((P, Q, R))

if len(pink_branch_data) != 2:
    raise RuntimeError(
        "Expected two visible pink branches, but found "
        f"{len(pink_branch_data)}. "
        "The current range-panel bounds may exclude a branch."
    )

# White halo first.
for branch_index, (P, Q, R) in enumerate(pink_branch_data):
    fig1.add_trace(
        go.Scatter3d(
            x=P,
            y=Q,
            z=R,
            mode="lines",
            line=dict(color="#ffffff", width=18),
            opacity=1.0,
            name="missing cubic halo",
            legendgroup="missing-cubic",
            showlegend=False,
            hoverinfo="skip",
            scene="scene2",
        )
    )

# Pink curve second, so it is the final geometry added to the figure.
for branch_index, (P, Q, R) in enumerate(pink_branch_data):
    fig1.add_trace(
        go.Scatter3d(
            x=P,
            y=Q,
            z=R,
            mode="lines",
            line=dict(color=PINK, width=12),
            opacity=1.0,
            name="missing cubic" if branch_index == 0 else "missing cubic ",
            legendgroup="missing-cubic",
            legendrank=1,
            showlegend=(branch_index == 0),
            hovertemplate=(
                "<b>missing triple-root curve</b>"
                "<br>P=%{x:.5g}"
                "<br>Q=%{y:.5g}"
                "<br>R=%{z:.5g}"
                "<extra></extra>"
            ),
            scene="scene2",
        )
    )

pink_visible_points = sum(
    int(np.count_nonzero(np.isfinite(P)))
    for P, Q, R in pink_branch_data
)
print(
    "Pink missing cubic added:",
    f"{len(pink_branch_data)} branches,",
    f"{pink_visible_points} visible sampled points",
)



## Freeze the windows and display

In [ ]:
# ============================================================
# Freeze panel windows and display
# ============================================================

fig1.layout.scene.xaxis.range = list(x_range)
fig1.layout.scene.yaxis.range = list(y_range)
fig1.layout.scene.zaxis.range = list(z_range)
fig1.layout.scene.xaxis.autorange = False
fig1.layout.scene.yaxis.autorange = False
fig1.layout.scene.zaxis.autorange = False

fig1.layout.scene2.xaxis.range = list(P_range)
fig1.layout.scene2.yaxis.range = list(Q_range)
fig1.layout.scene2.zaxis.range = list(R_range)
fig1.layout.scene2.xaxis.autorange = False
fig1.layout.scene2.yaxis.autorange = False
fig1.layout.scene2.zaxis.autorange = False

fig1.update_layout(
    title="Hamilton-Abel families with constant Q and constant R coordinate curves",
)

pink_traces = [
    trace for trace in fig1.data
    if str(getattr(trace, "name", "")).startswith("missing cubic")
    and str(getattr(trace, "name", "")) != "missing cubic halo"
]
assert len(pink_traces) == 2, len(pink_traces)

fig1.show(config={"scrollZoom": True, "displaylogo": False})


# Live direct-quadrature and mapped-mesh verification

The red routines are loaded from Artifact 24's original exact-area script. The Abel–Wick routines below are self-contained: they build each source-plane embedding, apply the polynomial map, form the Gram determinant, integrate its square root, and triangulate the same bounded disk.

In [ ]:
# Locate the project even if the depiction cells above were skipped.
def locate_artifact24_root() -> Path | None:
    here = Path.cwd().resolve()
    candidates = []
    for base in (here, *here.parents):
        candidates.extend([
            base,
            base / "24-jupyter-testing",
            base / "Artifacts" / "24-jupyter-testing",
        ])
    return next((p for p in candidates
                 if (p / "artifact24" / "geometry.py").is_file()), None)

LIVE_ROOT = locate_artifact24_root()
print("Artifact 24 root:", LIVE_ROOT)

red_geometry = None
if LIVE_ROOT is not None:
    script = LIVE_ROOT / "original_scripts" / "recompute_true_surface_area.py"
    spec = importlib.util.spec_from_file_location("artifact24_red_area", script)
    red_geometry = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(red_geometry)
    print("Loaded red quadrature and mesh routines")
else:
    print("Red live geometry unavailable here; precomputed red checks remain visible.")

In [ ]:
# Self-contained Abel–Wick geometry.
AW_X, AW_Y = sp.symbols("AW_X AW_Y", real=True)
AW_SQ3 = sp.sqrt(3)
AW_A = sp.Rational(2, 3) / AW_SQ3
AW_CENTROID = sp.Matrix([0, 0, sp.Rational(17, 4)])
AW_P = (
    sp.Matrix([0, 0, -3 * AW_SQ3 / 2]),
    sp.Matrix([AW_SQ3 / 3, -AW_SQ3 / 2, 3 * AW_SQ3 / 4]),
    sp.Matrix([-AW_SQ3 / 3, AW_SQ3 / 2, 3 * AW_SQ3 / 4]),
)
AW_K = sp.sqrt(sp.Rational(781, 13))
AW_Q = (
    sp.Matrix([-1, -sp.Rational(2, 3), 0]),
    sp.Matrix([-AW_K / 4, -AW_K / 6, 0]),
    sp.Matrix([-AW_K / 4, -AW_K / 6, 0]),
)
AW_NAMES = ("green", "yellow", "blue")


def aw_polynomial_map_symbolic(point):
    x, y, z = point
    xy = x * y
    return sp.Matrix([
        (1 + xy)**3 * z + y**2 * (1 + xy) * (4 + 3 * xy),
        y + 3*x*(1 + xy)**2*z + 3*x*y**2*(4 + 3*xy),
        2*x - 3*x**2*y - x**3*z,
    ])


def aw_local_range_map(family: int):
    p = AW_SQ3 + AW_X
    q = AW_Y / AW_SQ3
    source = AW_CENTROID + p * AW_P[family] + q * AW_Q[family]
    return sp.expand(aw_polynomial_map_symbolic(source))


_AW_NUMERIC = {}
def aw_numeric_family(family: int):
    if family in _AW_NUMERIC:
        return _AW_NUMERIC[family]
    range_map = aw_local_range_map(family)
    gx = range_map.diff(AW_X)
    gy = range_map.diff(AW_Y)
    density_squared = sp.expand(gx.cross(gy).dot(gx.cross(gy)))
    density = sp.sqrt(density_squared)
    center = sp.sqrt(sp.factor(density_squared.subs({AW_X: 0, AW_Y: 0})))
    rf = sp.lambdify((AW_X, AW_Y), range_map, "numpy")
    jf = sp.lambdify((AW_X, AW_Y), density, "numpy")
    def evaluate_range(x, y):
        values = np.asarray(rf(x, y), dtype=float)
        values = np.squeeze(values, axis=1)
        return np.moveaxis(values, 0, -1)
    _AW_NUMERIC[family] = (evaluate_range, jf, float(center))
    return _AW_NUMERIC[family]


def aw_boundary_radius(s: float, theta):
    theta = np.asarray(theta, dtype=float)
    cosine = np.cos(theta)
    a = float(AW_A)
    lo = np.zeros_like(cosine)
    hi = np.empty_like(cosine)
    nonnegative = cosine >= 0
    hi[nonnegative] = math.sqrt(s)
    negative = ~nonnegative
    if np.any(negative):
        c = cosine[negative]
        peak = -2.0 / (3.0 * a * c)
        trial = np.minimum(peak, np.full_like(peak, max(1.0, 2.0*math.sqrt(s))))
        values = trial**2 + a*c*trial**3
        for _ in range(60):
            mask = values < s
            if not np.any(mask):
                break
            trial[mask] = 0.5 * (trial[mask] + peak[mask])
            values = trial**2 + a*c*trial**3
        hi[negative] = trial
    for _ in range(72):
        mid = 0.5 * (lo + hi)
        values = mid**2 + a*cosine*mid**3
        lo = np.where(values < s, mid, lo)
        hi = np.where(values >= s, mid, hi)
    return 0.5 * (lo + hi)


def aw_direct_quadrature(family: int, s: float, ntheta=320, nrho=90):
    _, density, _ = aw_numeric_family(family)
    zt, wt = leggauss(ntheta)
    theta = math.pi * (zt + 1.0)
    wt = math.pi * wt
    zr, wr = leggauss(nrho)
    frac = 0.5 * (zr + 1.0)
    wr = 0.5 * wr
    radius = aw_boundary_radius(s, theta)
    r = frac[:, None] * radius[None, :]
    x = r * np.cos(theta)[None, :]
    y = r * np.sin(theta)[None, :]
    integrand = np.asarray(density(x, y), float) * frac[:, None] * radius[None, :]**2
    return float(np.sum(integrand * wr[:, None] * wt[None, :]))


def aw_structured_mesh(family: int, s: float, nr=24, ntheta=96):
    range_function, _, _ = aw_numeric_family(family)
    theta = np.linspace(0, 2*math.pi, ntheta, endpoint=False)
    radius = aw_boundary_radius(s, theta)
    vertices = [range_function(0.0, 0.0)]
    domain = [np.array([0.0, 0.0])]
    for ring in range(1, nr + 1):
        frac = ring / nr
        r = frac * radius
        xx = r*np.cos(theta); yy = r*np.sin(theta)
        vertices.extend(range_function(xx, yy))
        domain.extend(np.column_stack([xx, yy]))
    triangles = []
    for k in range(ntheta):
        triangles.append((0, 1+k, 1+(k+1) % ntheta))
    for ring in range(1, nr):
        inner = 1 + (ring-1)*ntheta
        outer = 1 + ring*ntheta
        for k in range(ntheta):
            k1 = (k+1) % ntheta
            triangles.append((inner+k, outer+k, outer+k1))
            triangles.append((inner+k, outer+k1, inner+k1))
    return np.asarray(domain), np.asarray(vertices), np.asarray(triangles, dtype=int)


def triangle_mesh_area(vertices, triangles):
    a = vertices[triangles[:, 0]]
    b = vertices[triangles[:, 1]]
    c = vertices[triangles[:, 2]]
    return float(0.5*np.linalg.norm(np.cross(b-a, c-a), axis=1).sum())

In [ ]:
# Recompute representative direct integrals and mesh convergence.
RUN_LIVE_VALIDATION = True

live_rows = []
if RUN_LIVE_VALIDATION:
    # Red: avoid m=0.70 because the center Taylor series is then outside its useful range.
    if red_geometry is not None:
        for m in (0.10, 0.40):
            direct = red_geometry.quadrature_area(m, ntheta=360, nrho=100)
            row = {
                "family": "red", "level": m,
                "direct_area": direct,
                "series_area": float(mapped_area_from_sequence("red", m)),
            }
            for nr, nt in ((12, 48), (24, 96), (40, 160)):
                _, vertices, triangles = red_geometry.structured_mesh(m, nr, nt)
                row[f"mesh_{len(triangles)}"] = red_geometry.mesh_area(vertices, triangles)
            live_rows.append(row)

    aw_levels = {
        0: (0.02, 0.05),
        1: (0.002, 0.005),
        2: (0.002, 0.005),
    }
    for family, levels in aw_levels.items():
        name = AW_NAMES[family]
        for s in levels:
            direct = aw_direct_quadrature(family, s)
            row = {
                "family": name, "level": s,
                "direct_area": direct,
                "series_area": float(mapped_area_from_sequence(name, s)),
            }
            for nr, nt in ((16, 64), (32, 128), (48, 192)):
                _, vertices, triangles = aw_structured_mesh(family, s, nr, nt)
                row[f"mesh_{len(triangles)}"] = triangle_mesh_area(vertices, triangles)
            live_rows.append(row)

live_validation = pd.DataFrame(live_rows)
if not live_validation.empty:
    live_validation["series_abs_error"] = abs(
        live_validation.direct_area - live_validation.series_area
    )
    mesh_columns = [c for c in live_validation if c.startswith("mesh_")]
    finest = mesh_columns[-1]
    live_validation["finest_mesh_abs_error"] = abs(
        live_validation.direct_area - live_validation[finest]
    )
    live_validation["finest_mesh_rel_error"] = (
        live_validation.finest_mesh_abs_error / live_validation.direct_area
    )
    display(live_validation)
    assert live_validation.series_abs_error.max() < 2e-10
    assert live_validation.finest_mesh_rel_error.max() < 8e-4
    print("Sequence → area → direct quadrature → mesh verification: PASS")

In [ ]:
# Mesh-convergence plot.
if not live_validation.empty:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.ravel()
    for ax, family in zip(axes, ("red", "green", "yellow", "blue")):
        group = live_validation[live_validation.family == family]
        mesh_columns = [c for c in group if c.startswith("mesh_")]
        for row in group.itertuples(index=False):
            ns = [int(c.split("_")[1]) for c in mesh_columns]
            errors = [abs(row.direct_area - getattr(row, c)) for c in mesh_columns]
            ax.loglog(ns, errors, marker="o", label=f"level={row.level:g}")
        ax.set_title(f"{family}: mapped-mesh convergence")
        ax.set_xlabel("triangles")
        ax.set_ylabel("absolute area error")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
# Four representative mapped meshes.
def add_mesh(ax, vertices, triangles, color, title):
    faces = vertices[triangles]
    poly = Poly3DCollection(
        faces, facecolors=color, edgecolors=(0.1, 0.1, 0.1, 0.25),
        linewidths=0.25, alpha=0.82
    )
    ax.add_collection3d(poly)
    mins = vertices.min(axis=0); maxs = vertices.max(axis=0)
    center = (mins + maxs)/2
    span = np.maximum(maxs-mins, 1e-8) * 0.58
    ax.set_xlim(center[0]-span[0], center[0]+span[0])
    ax.set_ylim(center[1]-span[1], center[1]+span[1])
    ax.set_zlim(center[2]-span[2], center[2]+span[2])
    ax.set_box_aspect(np.maximum(maxs-mins, 1e-8))
    ax.view_init(elev=23, azim=-61)
    ax.set_title(title)

fig = plt.figure(figsize=(14, 11))
plot_index = 1
if red_geometry is not None:
    _, rv, rt = red_geometry.structured_mesh(0.40, 24, 96)
    ax = fig.add_subplot(2, 2, plot_index, projection="3d")
    add_mesh(ax, rv, rt, "red", "red mapped disk, m=0.40")
    plot_index += 1
for family, level, color in [(0, 0.05, "green"),
                             (1, 0.005, "gold"),
                             (2, 0.005, "royalblue")]:
    _, vv, tt = aw_structured_mesh(family, level, 32, 128)
    ax = fig.add_subplot(2, 2, plot_index, projection="3d")
    add_mesh(ax, vv, tt, color, f"{AW_NAMES[family]} mapped lobe, s={level}")
    plot_index += 1
fig.suptitle("Mapped triangle meshes used in the area checks", fontsize=16)
fig.tight_layout()
plt.show()

## Reading the result

- The period sequences reconstruct mapped area by one integration in the level parameter.
- At the stated test levels, the reconstructed series and direct Gram-determinant quadrature agree essentially to floating-point precision.
- Ordinary mapped-triangle areas converge toward the same direct integral as the mesh is refined.
- Yellow and blue have separate projected meshes but the same complete period sequence through all 100 exact terms.
- The center expansions have finite convergence ranges. In particular, the red series should not be judged at $m=0.70$, and the green series should not be judged at $s=0.10$, even though the geometric areas themselves remain regular there.

The notebook therefore checks both the arithmetic statement and its geometric meaning:

\[
\boxed{
\text{integer-scaled period coefficients}
\longrightarrow
\text{area series}
\longrightarrow
\text{direct pullback area}
\longrightarrow
\text{mapped mesh area}.
}
\]